# HomeBase Concierge
### A tiered-trust personal concierge agent — Kaggle Capstone (5-Day Agents Course)

**Theme:** Concierge Agents — safe, useful personal assistants for planning, organization, and task management.

**Concepts demonstrated in this notebook:**
1. Multi-agent system built with an **ADK-style orchestrator** (single agent, skills loaded on demand)
2. **MCP servers** (Calendar, Email, Tasks, Weather) — mocked locally so this notebook runs with zero API keys, on a fresh Kaggle kernel
3. **Agent skills** with progressive disclosure
4. **Security features** — permission ladder (read/draft/act), PII redaction, human-in-the-loop approval, audit logging
5. **Evaluation** — a mechanical eval harness for routing accuracy, tier accuracy, and approval-gate integrity

**Architecture**

```
 User request -> ADK Orchestrator -> [Skills Library | Permission Ladder] -> MCP servers -> Audit & Approval
```

See the accompanying `README.md` for the full write-up. This notebook writes the real project source files to disk (`src/...`) cell by cell, then imports and runs them — so what you see executed here is exactly the shipped repo code, not a simplified rewrite.


In [ ]:
import os
os.makedirs('src/skills', exist_ok=True)
os.makedirs('src/mcp_servers', exist_ok=True)
os.makedirs('src/security', exist_ok=True)
for p in ['src', 'src/skills', 'src/mcp_servers', 'src/security']:
    open(os.path.join(p, '__init__.py'), 'a').close()
print('project scaffold ready')

project scaffold ready


## 1. Security layer

Every tool call is tagged `read` / `draft` / `act` before it reaches an MCP server. `act`-tier calls require an explicit approval callback (the human-in-the-loop gate). Every call — allowed, drafted, approved, or blocked — is written to an append-only audit log with PII-redacted payloads.

In [ ]:
%%writefile src/security/permissions.py
"""Permission ladder: read / draft / act.

Every tool call must be tagged with a tier before it reaches an MCP server.
- read  : always allowed, no side effects
- draft : always allowed, produces an artifact but commits nothing
- act   : requires explicit human approval before execution
"""
from enum import IntEnum
from dataclasses import dataclass, field
from typing import Callable, Optional


class Tier(IntEnum):
    READ = 0
    DRAFT = 1
    ACT = 2


@dataclass
class ToolCall:
    skill: str
    tool: str
    tier: Tier
    arguments: dict
    description: str  # human-readable summary shown for approval


class PermissionDenied(Exception):
    pass


class PermissionLadder:
    """Gates tool calls by tier. ACT calls require an approval callback."""

    def __init__(self, approve_fn: Optional[Callable[[ToolCall], bool]] = None, auto_approve: bool = False):
        # approve_fn(call) -> bool ; in a real deployment this would surface a UI prompt.
        self.approve_fn = approve_fn
        self.auto_approve = auto_approve  # only for scripted demo scenarios

    def authorize(self, call: ToolCall) -> bool:
        if call.tier in (Tier.READ, Tier.DRAFT):
            return True
        if call.tier == Tier.ACT:
            if self.auto_approve:
                return True
            if self.approve_fn is None:
                return False
            return bool(self.approve_fn(call))
        raise ValueError(f"Unknown tier: {call.tier}")


Writing src/security/permissions.py


In [ ]:
%%writefile src/security/pii_redaction.py
"""Lightweight PII redaction for logging and cross-skill payloads.

This is intentionally simple (regex-based) — good enough for a capstone demo.
A production system would use a proper PII-detection model/service.
"""
import re

EMAIL_RE = re.compile(r"[\w\.-]+@[\w\.-]+\.\w+")
PHONE_RE = re.compile(r"\b(\+?\d{1,2}[\s.-]?)?(\(?\d{3}\)?[\s.-]?)\d{3}[\s.-]?\d{4}\b")
ADDRESS_RE = re.compile(r"\d{1,5}\s+\w+(\s\w+){0,3}\s(Street|St|Avenue|Ave|Road|Rd|Boulevard|Blvd|Lane|Ln|Drive|Dr)\b", re.IGNORECASE)


def redact(text: str) -> str:
    if not text:
        return text
    text = EMAIL_RE.sub("[REDACTED_EMAIL]", text)
    text = PHONE_RE.sub("[REDACTED_PHONE]", text)
    text = ADDRESS_RE.sub("[REDACTED_ADDRESS]", text)
    return text


def redact_payload(payload: dict) -> dict:
    """Recursively redact string values in a dict, for safe audit logging."""
    out = {}
    for k, v in payload.items():
        if isinstance(v, str):
            out[k] = redact(v)
        elif isinstance(v, dict):
            out[k] = redact_payload(v)
        else:
            out[k] = v
    return out


Writing src/security/pii_redaction.py


In [ ]:
%%writefile src/security/audit.py
"""Append-only audit log for every tool call — allowed, drafted, or blocked."""
import time
import hashlib
import json
from dataclasses import dataclass, asdict
from typing import Any, List

from .pii_redaction import redact_payload


@dataclass
class AuditEntry:
    timestamp: float
    skill: str
    tool: str
    tier: str
    decision: str  # "allowed" | "blocked" | "approved" | "rejected"
    payload_redacted: dict
    payload_hash: str


class AuditLog:
    def __init__(self):
        self._entries: List[AuditEntry] = []

    def record(self, skill: str, tool: str, tier: str, decision: str, payload: dict):
        redacted = redact_payload(payload)
        payload_hash = hashlib.sha256(json.dumps(payload, sort_keys=True, default=str).encode()).hexdigest()[:12]
        entry = AuditEntry(
            timestamp=time.time(),
            skill=skill,
            tool=tool,
            tier=tier,
            decision=decision,
            payload_redacted=redacted,
            payload_hash=payload_hash,
        )
        self._entries.append(entry)
        return entry

    def entries(self) -> List[dict]:
        return [asdict(e) for e in self._entries]

    def print_log(self):
        for e in self._entries:
            t = time.strftime("%H:%M:%S", time.localtime(e.timestamp))
            print(f"[{t}] {e.decision:9s} | {e.skill:15s} | {e.tool:20s} | tier={e.tier:5s} | hash={e.payload_hash}")


Writing src/security/audit.py


## 2. MCP servers (mocked locally)

These follow the same call/response shape a real MCP tool call would use — name in, structured result out. Swapping in real MCP servers (Google Calendar API, Gmail API, a tasks backend, a weather API) requires **no changes** to the orchestrator, skills, or security layer — only these four files change.

In [ ]:
%%writefile src/mcp_servers/calendar_mcp.py
"""Local stand-in for a Calendar MCP server.

Mirrors the shape of a real MCP tool call: name in, structured result out.
Swap this for a real Google Calendar MCP server without touching the
orchestrator, skills, or security layer.
"""
from datetime import datetime, timedelta

_EVENTS = [
    {"id": "evt1", "title": "Team sync", "start": "2026-07-06T15:00:00", "end": "2026-07-06T15:30:00"},
    {"id": "evt2", "title": "Dentist", "start": "2026-07-08T09:00:00", "end": "2026-07-08T10:00:00"},
]


def list_events(start: str = None, end: str = None):
    """read-tier: list events in range."""
    return {"events": _EVENTS}


def propose_reschedule(event_id: str, new_start: str, new_end: str):
    """draft-tier: build a proposed change, do not apply it."""
    event = next((e for e in _EVENTS if e["id"] == event_id), None)
    if not event:
        return {"error": f"event {event_id} not found"}
    return {
        "draft": {
            "event_id": event_id,
            "title": event["title"],
            "old_start": event["start"],
            "new_start": new_start,
            "new_end": new_end,
        }
    }


def commit_reschedule(event_id: str, new_start: str, new_end: str):
    """act-tier: actually move the event. Must go through the permission ladder first."""
    event = next((e for e in _EVENTS if e["id"] == event_id), None)
    if not event:
        return {"error": f"event {event_id} not found"}
    event["start"], event["end"] = new_start, new_end
    return {"status": "committed", "event": event}


Writing src/mcp_servers/calendar_mcp.py


In [ ]:
%%writefile src/mcp_servers/email_mcp.py
"""Local stand-in for an Email MCP server.

Deliberately supports draft creation but NOT sending — HomeBase never
sends email autonomously in this project; sending is treated as
out-of-scope act-tier and always requires a real client / human hand-off.
"""
_DRAFTS = []


def create_draft(to: str, subject: str, body: str):
    """draft-tier: create an email draft, do not send."""
    draft = {"id": f"draft{len(_DRAFTS)+1}", "to": to, "subject": subject, "body": body}
    _DRAFTS.append(draft)
    return {"draft": draft}


def list_drafts():
    """read-tier."""
    return {"drafts": _DRAFTS}


Writing src/mcp_servers/email_mcp.py


In [ ]:
%%writefile src/mcp_servers/tasks_mcp.py
"""Local stand-in for a Tasks/Notes MCP server (to-dos, shopping lists, expense notes)."""
_TASKS = [
    {"id": "t1", "title": "Buy birthday gift", "done": False, "list": "errands"},
]
_EXPENSES = [
    {"category": "takeout", "amount": 42.50, "date": "2026-07-01"},
    {"category": "takeout", "amount": 18.00, "date": "2026-06-27"},
    {"category": "groceries", "amount": 96.30, "date": "2026-06-29"},
]


def list_tasks(list_name: str = None):
    """read-tier."""
    if list_name:
        return {"tasks": [t for t in _TASKS if t["list"] == list_name]}
    return {"tasks": _TASKS}


def propose_task_list(items: list, list_name: str):
    """draft-tier: propose a new set of tasks (e.g. a shopping list), do not save."""
    return {"draft": {"list": list_name, "items": items}}


def commit_task_list(items: list, list_name: str):
    """act-tier: actually save the tasks."""
    for item in items:
        _TASKS.append({"id": f"t{len(_TASKS)+1}", "title": item, "done": False, "list": list_name})
    return {"status": "committed", "count": len(items)}


def summarize_expenses(category: str = None, month: str = None):
    """read-tier."""
    rows = _EXPENSES
    if category:
        rows = [r for r in rows if r["category"] == category]
    total = sum(r["amount"] for r in rows)
    return {"total": round(total, 2), "rows": rows}


Writing src/mcp_servers/tasks_mcp.py


In [ ]:
%%writefile src/mcp_servers/weather_mcp.py
"""Local stand-in for a Weather MCP server, used for trip planning."""
_FORECASTS = {
    "Boston": [{"date": "2026-07-08", "high_f": 72, "low_f": 58, "condition": "rain"}],
    "Ho Chi Minh City": [{"date": "2026-07-08", "high_f": 91, "low_f": 79, "condition": "thunderstorms"}],
}


def get_forecast(city: str, date: str = None):
    """read-tier."""
    return {"city": city, "forecast": _FORECASTS.get(city, [{"date": date, "high_f": 75, "low_f": 60, "condition": "unknown"}])}


Writing src/mcp_servers/weather_mcp.py


## 3. Skills library (progressive disclosure)

Each skill is a small, self-contained handler with its own manifest (trigger keywords, MCP servers used, default max permission tier). The orchestrator only needs the trigger keywords to route — a skill's full logic stays out of the orchestrator's context until it's actually loaded.

In [ ]:
%%writefile src/skills/scheduling.py
"""Scheduling skill: read/reschedule calendar events.

Manifest (what the orchestrator checks before loading this skill in full):
  triggers: "calendar", "schedule", "reschedule", "move my", "what's on"
  mcp_servers: [calendar_mcp]
  default_max_tier: draft   (committing a reschedule is act-tier and needs approval)
"""
from ..mcp_servers import calendar_mcp
from ..security.permissions import ToolCall, Tier

NAME = "scheduling"


def handle(request: str, ladder, audit):
    request_l = request.lower()

    if "move" in request_l or "reschedule" in request_l:
        # Step 1: propose the change (draft-tier, always allowed)
        call = ToolCall(NAME, "propose_reschedule", Tier.DRAFT,
                         {"event_id": "evt1", "new_start": "2026-07-09T10:00:00", "new_end": "2026-07-09T10:30:00"},
                         "Draft: move 'Team sync' to Thu Jul 9, 10:00am")
        ladder.authorize(call)
        result = calendar_mcp.propose_reschedule(**call.arguments)
        audit.record(NAME, call.tool, call.tier.name, "allowed", call.arguments)

        # Step 2: committing the change is act-tier -> needs approval
        commit_call = ToolCall(NAME, "commit_reschedule", Tier.ACT, call.arguments,
                                "Move 'Team sync' from Mon 3:00pm to Thu 10:00am — confirm?")
        approved = ladder.authorize(commit_call)
        if approved:
            commit_result = calendar_mcp.commit_reschedule(**commit_call.arguments)
            audit.record(NAME, commit_call.tool, commit_call.tier.name, "approved", commit_call.arguments)
            return f"Done — moved to {commit_call.arguments['new_start']}. (draft: {result['draft']})"
        else:
            audit.record(NAME, commit_call.tool, commit_call.tier.name, "blocked", commit_call.arguments)
            return f"I've drafted the change but need your OK before moving it: {result['draft']}"

    # default: read-only lookup
    call = ToolCall(NAME, "list_events", Tier.READ, {}, "List upcoming events")
    ladder.authorize(call)
    result = calendar_mcp.list_events()
    audit.record(NAME, call.tool, call.tier.name, "allowed", call.arguments)
    return f"Upcoming: {result['events']}"


Writing src/skills/scheduling.py


In [ ]:
%%writefile src/skills/meal_planning.py
"""Meal planning skill: generate a weekly plan + shopping list (draft-tier).

Manifest:
  triggers: "dinner", "meal plan", "grocery list", "what should I cook"
  mcp_servers: [tasks_mcp]
  default_max_tier: draft
"""
from ..mcp_servers import tasks_mcp
from ..security.permissions import ToolCall, Tier

NAME = "meal_planning"

_SAMPLE_PLAN = ["Mon: stir-fry veggies + tofu", "Tue: sheet-pan chicken", "Wed: lentil soup",
                "Thu: leftovers", "Fri: pizza night"]
_SHOPPING_LIST = ["tofu", "mixed veggies", "chicken thighs", "lentils", "pizza dough"]


def handle(request: str, ladder, audit):
    call = ToolCall(NAME, "propose_task_list", Tier.DRAFT,
                     {"items": _SHOPPING_LIST, "list_name": "groceries"},
                     "Draft shopping list for this week's meal plan")
    ladder.authorize(call)
    result = tasks_mcp.propose_task_list(**call.arguments)
    audit.record(NAME, call.tool, call.tier.name, "allowed", call.arguments)
    plan_str = "\n".join(_SAMPLE_PLAN)
    return f"Here's a plan for the week:\n{plan_str}\n\nDraft shopping list (not saved yet): {result['draft']['items']}"


Writing src/skills/meal_planning.py


In [ ]:
%%writefile src/skills/travel_prep.py
"""Travel prep skill: forecast-driven packing advice (read-tier only).

Manifest:
  triggers: "pack", "trip", "traveling to", "weather in"
  mcp_servers: [weather_mcp, calendar_mcp]
  default_max_tier: read
"""
from ..mcp_servers import weather_mcp
from ..security.permissions import ToolCall, Tier

NAME = "travel_prep"


def handle(request: str, ladder, audit, city: str = "Boston"):
    call = ToolCall(NAME, "get_forecast", Tier.READ, {"city": city}, f"Look up forecast for {city}")
    ladder.authorize(call)
    result = weather_mcp.get_forecast(**call.arguments)
    audit.record(NAME, call.tool, call.tier.name, "allowed", call.arguments)
    day = result["forecast"][0]
    advice = "Pack a rain jacket and layers." if "rain" in day["condition"] else "Light layers should be fine."
    return f"{city} on {day['date']}: {day['low_f']}-{day['high_f']}°F, {day['condition']}. {advice}"


Writing src/skills/travel_prep.py


In [ ]:
%%writefile src/skills/budgeting.py
"""Budgeting skill: summarize logged expenses (read-tier only, never initiates spend).

Manifest:
  triggers: "how much did I spend", "budget", "expenses"
  mcp_servers: [tasks_mcp]
  default_max_tier: read
"""
from ..mcp_servers import tasks_mcp
from ..security.permissions import ToolCall, Tier

NAME = "budgeting"


def handle(request: str, ladder, audit, category: str = "takeout"):
    call = ToolCall(NAME, "summarize_expenses", Tier.READ, {"category": category}, f"Summarize {category} spend")
    ladder.authorize(call)
    result = tasks_mcp.summarize_expenses(**call.arguments)
    audit.record(NAME, call.tool, call.tier.name, "allowed", call.arguments)
    return f"You've spent ${result['total']} on {category} recently, across {len(result['rows'])} purchases."


Writing src/skills/budgeting.py


In [ ]:
%%writefile src/skills/correspondence.py
"""Correspondence skill: draft emails. Sending is out of scope by design —
HomeBase is never permitted to send email autonomously in this project.

Manifest:
  triggers: "email", "reply to", "draft a note to"
  mcp_servers: [email_mcp]
  default_max_tier: draft
"""
from ..mcp_servers import email_mcp
from ..security.permissions import ToolCall, Tier

NAME = "correspondence"


def handle(request: str, ladder, audit, to: str = "sam@example.com"):
    subject = "Re: Saturday"
    body = "Hi Sam — Saturday no longer works for me, could we push to Sunday afternoon instead?"
    call = ToolCall(NAME, "create_draft", Tier.DRAFT, {"to": to, "subject": subject, "body": body},
                     f"Draft email to {to}")
    ladder.authorize(call)
    result = email_mcp.create_draft(**call.arguments)
    audit.record(NAME, call.tool, call.tier.name, "allowed", call.arguments)
    return f"Drafted (not sent): {result['draft']}"


Writing src/skills/correspondence.py


## 4. ADK-style orchestrator (with a real LLM routing option)

The orchestrator ships with **two interchangeable routers**:
- `router="keyword"` (default) — deterministic, no API key needed, used for the reproducible demo and eval harness below.
- `router="llm"` — a real Gemini call decides the skill, which is the realistic ADK-style pattern. If no `GEMINI_API_KEY`/`GOOGLE_API_KEY` is set (as on a fresh Kaggle kernel with no secret added), or the `google-genai` package isn't installed, or the call fails for any reason, it **automatically falls back to keyword routing** and logs why — so this notebook still runs end-to-end with zero setup, while supporting a real LLM router the moment you add a key.

In [ ]:
%%writefile src/routing_llm.py
"""LLM-based intent router (optional upgrade over keyword routing).

Uses the Google Gen AI SDK (`google-genai`, the current unified SDK for the
Gemini API) to have a real model decide which skill should handle a
request — the "ADK-style" routing implied in the architecture diagram,
where a real orchestrator agent would use an LLM call rather than
keyword matching.

Falls back to keyword routing automatically if no API key is configured,
the `google-genai` package isn't installed, or the call fails for any
reason — so a notebook using this router still runs end-to-end on a
fresh kernel with no key set (it just logs that it fell back).

For local/dev use, this module will auto-load a `.env` file from the
project root if `python-dotenv` is installed (optional dependency) —
put `GEMINI_API_KEY=your-key-here` in a `.env` file next to `README.md`
and it will be picked up automatically. On Kaggle, use Add-ons -> Secrets
instead (a `.env` file won't be present on a fresh kernel).
"""
import os

try:
    from dotenv import load_dotenv, find_dotenv
    load_dotenv(find_dotenv(usecwd=True))
except ImportError:
    pass  # python-dotenv not installed — fine, just relies on real env vars / Kaggle secrets

SKILL_NAMES = ["scheduling", "meal_planning", "travel_prep", "budgeting", "correspondence", "none"]

_SYSTEM_INSTRUCTION = (
    "You are the routing layer of a personal concierge agent called HomeBase. "
    "Given a user's request, decide which single skill should handle it.\n"
    "Valid skills:\n"
    "- scheduling: calendar lookups, moving/rescheduling meetings\n"
    "- meal_planning: dinner plans, grocery/shopping lists\n"
    "- travel_prep: weather/packing advice for a trip\n"
    "- budgeting: spending summaries, expense questions\n"
    "- correspondence: drafting an email or reply\n"
    "If none clearly apply, answer 'none'.\n"
    "Respond with ONLY the skill name, lowercase, nothing else — no punctuation, no explanation."
)


class LLMRouterUnavailable(Exception):
    """Raised when the LLM router can't be used; caller should fall back to keyword routing."""


def llm_route(request: str, api_key: str = None, model: str = "gemini-2.5-flash") -> str:
    """Return a skill name (or 'none') using a real Gemini call.

    Raises LLMRouterUnavailable (never a raw SDK exception) so callers can
    catch one exception type and fall back cleanly.
    """
    api_key = api_key or os.environ.get("GEMINI_API_KEY") or os.environ.get("GOOGLE_API_KEY")
    if not api_key:
        raise LLMRouterUnavailable("no GEMINI_API_KEY / GOOGLE_API_KEY configured")

    try:
        from google import genai
        from google.genai import types
    except ImportError as e:
        raise LLMRouterUnavailable(f"google-genai is not installed ({e}); run: pip install google-genai")

    try:
        client = genai.Client(api_key=api_key)
        response = client.models.generate_content(
            model=model,
            contents=request,
            config=types.GenerateContentConfig(
                system_instruction=_SYSTEM_INSTRUCTION,
                temperature=0,
            ),
        )
        answer = (response.text or "").strip().lower()
    except Exception as e:
        raise LLMRouterUnavailable(f"Gemini call failed: {e}")

    for name in SKILL_NAMES:
        if name in answer:
            return name
    return "none"


Writing src/routing_llm.py


In [ ]:
%%writefile src/orchestrator.py
"""ADK-style orchestrator agent.

In a real deployment this would be an `Agent` from google-adk with an LLM
doing the routing. Here routing is keyword-based so the notebook runs
deterministically without an API key — the *shape* (route -> load skill on
demand -> pass shared ladder/audit) is what maps onto the real ADK pattern.
"""
from .skills import scheduling, meal_planning, travel_prep, budgeting, correspondence
from .security.permissions import PermissionLadder
from .security.audit import AuditLog
from .routing_llm import llm_route, LLMRouterUnavailable

SKILL_MANIFEST = [
    (scheduling, ["calendar", "schedule", "reschedule", "move my", "what's on"]),
    (meal_planning, ["dinner", "meal plan", "grocery", "cook"]),
    (travel_prep, ["pack", "trip", "traveling", "weather"]),
    (budgeting, ["spend", "budget", "expense"]),
    (correspondence, ["email", "reply to", "draft a note"]),
]

SKILL_BY_NAME = {skill_module.NAME: skill_module for skill_module, _ in SKILL_MANIFEST}


class HomeBaseOrchestrator:
    def __init__(self, approve_fn=None, auto_approve=False,
                 router: str = "keyword", gemini_api_key: str = None, gemini_model: str = "gemini-2.5-flash"):
        """
        router: "keyword" (default, deterministic, no API key needed) or
                "llm" (real Gemini call decides the skill; falls back to
                keyword routing automatically if no key/package/call fails).
        """
        self.ladder = PermissionLadder(approve_fn=approve_fn, auto_approve=auto_approve)
        self.audit = AuditLog()
        self.router = router
        self.gemini_api_key = gemini_api_key
        self.gemini_model = gemini_model

    def _route_by_keyword(self, request: str):
        r = request.lower()
        for skill_module, keywords in SKILL_MANIFEST:
            if any(k in r for k in keywords):
                return skill_module
        return None

    def route(self, request: str):
        """Progressive disclosure: only decide which skill module to load."""
        if self.router == "llm":
            try:
                name = llm_route(request, api_key=self.gemini_api_key, model=self.gemini_model)
                return SKILL_BY_NAME.get(name)  # None for "none" or an unrecognized name
            except LLMRouterUnavailable as e:
                print(f"[router] LLM routing unavailable ({e}) — falling back to keyword routing for this request")
        return self._route_by_keyword(request)

    def handle(self, request: str, **kwargs):
        skill_module = self.route(request)
        if skill_module is None:
            return "I'm not sure which skill handles that yet — try asking about scheduling, meals, travel, budget, or email."
        return skill_module.handle(request, self.ladder, self.audit, **kwargs)


Writing src/orchestrator.py


In [ ]:
%%writefile src/evaluation.py
"""Mechanical evaluation harness (no human judgment needed):
1. Skill-routing accuracy
2. Permission-tier accuracy for the first tool call issued per request
3. Approval-gate integrity: no ACT call ever executes without a recorded approval
"""
from .orchestrator import HomeBaseOrchestrator

ROUTING_CASES = [
    ("what's on my calendar this week", "scheduling"),
    ("move my 3pm to tomorrow", "scheduling"),
    ("plan dinner for the week", "meal_planning"),
    ("grocery list please", "meal_planning"),
    ("what's the weather like on my trip to Boston", "travel_prep"),
    ("how much did I spend on takeout", "budgeting"),
    ("draft a reply to Sam", "correspondence"),
]

TIER_CASES = [
    ("what's on my calendar this week", "READ"),
    ("plan dinner for the week", "DRAFT"),
    ("how much did I spend on takeout", "READ"),
    ("draft a reply to Sam", "DRAFT"),
]


def run_routing_eval():
    orch = HomeBaseOrchestrator(auto_approve=True)
    correct = 0
    for request, expected in ROUTING_CASES:
        skill = orch.route(request)
        got = skill.NAME if skill else None
        ok = got == expected
        correct += ok
        print(f"{'PASS' if ok else 'FAIL'}  '{request}' -> expected={expected} got={got}")
    print(f"\nRouting accuracy: {correct}/{len(ROUTING_CASES)}")
    return correct / len(ROUTING_CASES)


def run_tier_eval():
    correct = 0
    for request, expected_tier in TIER_CASES:
        orch = HomeBaseOrchestrator(auto_approve=True)
        orch.handle(request)
        first_entry = orch.audit.entries()[0]
        ok = first_entry["tier"] == expected_tier
        correct += ok
        print(f"{'PASS' if ok else 'FAIL'}  '{request}' -> expected={expected_tier} got={first_entry['tier']}")
    print(f"\nTier accuracy: {correct}/{len(TIER_CASES)}")
    return correct / len(TIER_CASES)


def run_approval_gate_eval():
    """Confirm that when approval is withheld, no ACT-tier call is ever marked 'approved'."""
    orch = HomeBaseOrchestrator(approve_fn=lambda call: False)  # always reject
    orch.handle("move my 3pm to tomorrow")
    entries = orch.audit.entries()
    act_entries = [e for e in entries if e["tier"] == "ACT"]
    leaked = [e for e in act_entries if e["decision"] == "approved"]
    ok = len(leaked) == 0
    print(f"{'PASS' if ok else 'FAIL'}  ACT calls executed without approval: {len(leaked)} (should be 0)")
    return ok


Writing src/evaluation.py


In [ ]:
import importlib, sys
for m in list(sys.modules):
    if m.startswith('src'):
        del sys.modules[m]
from src.orchestrator import HomeBaseOrchestrator
print('orchestrator imported')

orchestrator imported


### Optional: try the real LLM router
Set a Gemini API key as a Kaggle secret (`Add-ons -> Secrets`) named `GEMINI_API_KEY`, or just set the environment variable below, then re-run this cell. With no key set, you'll see the fallback message and the keyword result — nothing breaks either way.

In [ ]:
# pip install google-genai   # uncomment if you want to try the real LLM router
import os
# os.environ['GEMINI_API_KEY'] = 'your-key-here'  # uncomment and fill in to try a real Gemini call

orch_llm = HomeBaseOrchestrator(router='llm', auto_approve=False)
print(orch_llm.handle('how much did I spend on takeout?'))


[router] LLM routing unavailable (no GEMINI_API_KEY / GOOGLE_API_KEY configured) — falling back to keyword routing for this request
You've spent $60.5 on takeout recently, across 2 purchases.


## 5. Demo: everyday requests (read/draft tiers, auto-flow)

These requests only ever need `read` or `draft` tier calls, so nothing here requires human approval — but everything is still logged.

In [ ]:
orch = HomeBaseOrchestrator(auto_approve=False)  # no approve_fn needed yet — no ACT calls in this batch

print(orch.handle("what should I cook for dinner this week?"))
print()
print(orch.handle("how much did I spend on takeout?"))
print()
print(orch.handle("will I need a jacket on my trip to Boston?", city="Boston"))
print()
print(orch.handle("draft a reply to Sam about Saturday"))


Here's a plan for the week:
Mon: stir-fry veggies + tofu
Tue: sheet-pan chicken
Wed: lentil soup
Thu: leftovers
Fri: pizza night

Draft shopping list (not saved yet): ['tofu', 'mixed veggies', 'chicken thighs', 'lentils', 'pizza dough']

You've spent $60.5 on takeout recently, across 2 purchases.

Boston on 2026-07-08: 58-72°F, rain. Pack a rain jacket and layers.

Drafted (not sent): {'id': 'draft1', 'to': 'sam@example.com', 'subject': 'Re: Saturday', 'body': 'Hi Sam — Saturday no longer works for me, could we push to Sunday afternoon instead?'}


## 6. Demo: human-in-the-loop approval (act-tier)

Rescheduling a calendar event actually *commits* a change, so it's `act`-tier. The orchestrator drafts the change first (always allowed), then pauses for approval before committing.

In [ ]:
def approve(call):
    print(f"[APPROVAL REQUESTED] {call.description}")
    decision = True  # in a real UI this would be a user click; auto-approving here for the notebook demo
    print(f"[APPROVAL RESULT] {'approved' if decision else 'rejected'}")
    return decision

orch2 = HomeBaseOrchestrator(approve_fn=approve)
print(orch2.handle("can you move my 3pm meeting?"))


[APPROVAL REQUESTED] Move 'Team sync' from Mon 3:00pm to Thu 10:00am — confirm?
[APPROVAL RESULT] approved
Done — moved to 2026-07-09T10:00:00. (draft: {'event_id': 'evt1', 'title': 'Team sync', 'old_start': '2026-07-06T15:00:00', 'new_start': '2026-07-09T10:00:00', 'new_end': '2026-07-09T10:30:00'})


## 7. Demo: the permission ladder correctly *blocks* an action

This is the important negative case for the submission checklist: when approval is withheld, the `act`-tier call must never execute, and the audit log must reflect that it was blocked, not silently skipped.

In [ ]:
orch3 = HomeBaseOrchestrator(approve_fn=lambda call: False)  # simulate the user declining
print(orch3.handle("can you move my 3pm meeting?"))
print()
print('--- audit log ---')
orch3.audit.print_log()


I've drafted the change but need your OK before moving it: {'event_id': 'evt1', 'title': 'Team sync', 'old_start': '2026-07-09T10:00:00', 'new_start': '2026-07-09T10:00:00', 'new_end': '2026-07-09T10:30:00'}

--- audit log ---
[14:48:21] allowed   | scheduling      | propose_reschedule   | tier=DRAFT | hash=e129f0c13206
[14:48:21] blocked   | scheduling      | commit_reschedule    | tier=ACT   | hash=e129f0c13206


## 8. Audit log inspection

Full, human-readable audit trail for the human-in-the-loop demo session (section 6) — every call, its tier, and its decision.

In [ ]:
orch2.audit.print_log()

[14:48:21] allowed   | scheduling      | propose_reschedule   | tier=DRAFT | hash=e129f0c13206


[14:48:21] approved  | scheduling      | commit_reschedule    | tier=ACT   | hash=e129f0c13206


## 9. PII redaction check

A quick adversarial check: if a request contains an email address or phone number, the audit log must never store it in the clear.

In [ ]:
from src.security.pii_redaction import redact, redact_payload

sample = {"to": "sam.jones@example.com", "note": "call me at 555-123-4567 about 12 Main Street"}
print('raw:     ', sample)
print('redacted:', redact_payload(sample))


raw:      {'to': 'sam.jones@example.com', 'note': 'call me at 555-123-4567 about 12 Main Street'}
redacted: {'to': '[REDACTED_EMAIL]', 'note': 'call me at [REDACTED_PHONE] about [REDACTED_ADDRESS]'}


## 10. Evaluation harness

Three mechanical checks (no human judgment needed, fully reproducible):
1. **Routing accuracy** — does the orchestrator load the right skill?
2. **Tier accuracy** — is the first tool call for a request tagged with the expected permission tier?
3. **Approval-gate integrity** — is it ever possible for an `act`-tier call to execute without a recorded approval? (Must always be 0.)

In [ ]:
from src.evaluation import run_routing_eval, run_tier_eval, run_approval_gate_eval

print('=== Routing accuracy ===')
routing_score = run_routing_eval()
print()
print('=== Tier accuracy ===')
tier_score = run_tier_eval()
print()
print('=== Approval-gate integrity ===')
gate_ok = run_approval_gate_eval()
print()
print(f'Summary: routing={routing_score:.0%}, tier={tier_score:.0%}, approval_gate_safe={gate_ok}')


=== Routing accuracy ===
PASS  'what's on my calendar this week' -> expected=scheduling got=scheduling
PASS  'move my 3pm to tomorrow' -> expected=scheduling got=scheduling
PASS  'plan dinner for the week' -> expected=meal_planning got=meal_planning
PASS  'grocery list please' -> expected=meal_planning got=meal_planning
PASS  'what's the weather like on my trip to Boston' -> expected=travel_prep got=travel_prep
PASS  'how much did I spend on takeout' -> expected=budgeting got=budgeting
PASS  'draft a reply to Sam' -> expected=correspondence got=correspondence

Routing accuracy: 7/7

=== Tier accuracy ===
PASS  'what's on my calendar this week' -> expected=READ got=READ
PASS  'plan dinner for the week' -> expected=DRAFT got=DRAFT
PASS  'how much did I spend on takeout' -> expected=READ got=READ
PASS  'draft a reply to Sam' -> expected=DRAFT got=DRAFT

Tier accuracy: 4/4

=== Approval-gate integrity ===
PASS  ACT calls executed without approval: 0 (should be 0)

Summary: routing=100%, ti

## 11. What's mocked vs. real, and next steps

To keep this notebook reproducible on a fresh Kaggle kernel with zero API keys, the MCP servers here are **local stand-ins** that follow the same call/response shape a real MCP tool call would use. Swapping in real `google-adk` orchestration and live MCP servers (Google Calendar API, Gmail API) is a drop-in replacement — the orchestrator, skills, and security layer do not need to change, only the four `mcp_servers/*.py` implementations.

**With more time, I'd add:**
- Real ADK `Agent`/`LlmAgent` routing (LLM-based intent classification instead of keyword matching)
- Live OAuth-backed Google Calendar / Gmail MCP servers
- Persistent audit log storage (currently in-memory per session)
- A real approval UI instead of a Python callback
- Human-scored evaluation of drafted content quality (meal plans, email drafts), on top of the mechanical checks above

**Submission checklist** (see `README.md` for the full version):
- [x] Runs top-to-bottom with no errors on a fresh kernel
- [x] 5 course concepts demonstrated (ADK orchestrator, MCP servers, skills, security, evaluation)
- [x] Architecture diagram included (README)
- [x] At least one correctly-blocked/escalated action (Section 7)
- [x] Audit log shown for a full session (Section 8)
- [x] "What's mocked vs. real" + next steps (this section)
